In [ ]:
import wrds
import pandas as pd

conn = wrds.Connection()

In [ ]:
comp_names = conn.raw_sql("""
    SELECT gvkey, conm
    FROM comp.company
""")

comp_names.shape

In [ ]:
comp_names.to_csv('comp_names.csv', index=False)

In [ ]:
comp_tickers = conn.raw_sql("""
    SELECT gvkey, iid, ticker, exchg
    FROM comp.sec
""")

comp_tickers.head()

In [ ]:
gvkey_permno = conn.raw_sql("""
    SELECT gvkey, lpermno as permno, lpermco as permco
    FROM crsp.ccmxpf_linktable
    WHERE linktype IN ('LU', 'LC') -- Active links
""")

crsp_tickers = conn.raw_sql("""
    SELECT permno, ticker
    FROM crsp.stocknames
""")

# Merge the tables to get GVKEY → PERMNO → Ticker
gvkey_to_ticker = gvkey_permno.merge(crsp_tickers, on="permno", how="left")

gvkey_to_ticker.head()

In [ ]:
gvkey_to_ticker.shape

In [ ]:
gvkey_to_ticker.to_csv("gvkey_to_ticker.csv")

In [ ]:
#Find apple in comp_names
comp_names[comp_names["conm"].str.contains("nvidia", case=False)]

In [ ]:
gvkey_to_ticker[gvkey_to_ticker["ticker"].str.contains("BSC", case=False)]

In [ ]:
comp_names[comp_names["gvkey"] == "011818"]

In [ ]:
# 🏢 Create Comprehensive GVKEY to Sector/Industry Mapping
# ========================================================

print("🔍 Creating comprehensive GVKEY mapping with sector/industry data...")

# Enhanced query to get GVKEY, sector, industry, and ticker mapping
gvkey_mapping = conn.raw_sql("""
    SELECT
        c.gvkey,
        c.conm AS company_name,
        c.sic,
        c.gsector AS gics_sector,
        c.gind AS gics_industry, 
        c.gsubind AS gics_subindustry,
        c.naicsh AS naics_code,
        l.lpermno AS permno,
        l.lpermco AS permco,
        s.ticker,
        s.exchcd AS exchange_code,
        -- Add sector names for better readability
        CASE 
            WHEN c.gsector = '10' THEN 'Energy'
            WHEN c.gsector = '15' THEN 'Materials'
            WHEN c.gsector = '20' THEN 'Industrials'
            WHEN c.gsector = '25' THEN 'Consumer Discretionary'
            WHEN c.gsector = '30' THEN 'Consumer Staples'
            WHEN c.gsector = '35' THEN 'Health Care'
            WHEN c.gsector = '40' THEN 'Financials'
            WHEN c.gsector = '45' THEN 'Information Technology'
            WHEN c.gsector = '50' THEN 'Communication Services'
            WHEN c.gsector = '55' THEN 'Utilities'
            WHEN c.gsector = '60' THEN 'Real Estate'
            ELSE 'Other'
        END AS sector_name
    FROM comp.company AS c
    LEFT JOIN crsp.ccmxpf_linktable AS l
        ON c.gvkey = l.gvkey
        AND l.linktype IN ('LU', 'LC')
        AND l.usedflag = 1
    LEFT JOIN crsp.stocknames AS s
        ON l.lpermno = s.permno
    WHERE c.gvkey IS NOT NULL
""")

print(f"✅ Retrieved {len(gvkey_mapping)} company mappings")
print(f"📊 Shape: {gvkey_mapping.shape}")
print(f"🏢 Unique companies: {gvkey_mapping['gvkey'].nunique()}")
print(f"🎯 With tickers: {gvkey_mapping['ticker'].notna().sum()}")

# Display sample data
print(f"\n📋 Sample mapping data:")
gvkey_mapping.head()